# Conformal geometry: a dielectric-lined pillbox cavity

By default, Wakis assigns one material per mesh cell (**staircase** geometry). A curved surface is then replaced by the faces of the cells it passes through, which changes the geometry by up to one cell size. For curved conductors and dielectrics this shifts resonance frequencies and limits how fast results converge with mesh refinement.

With `geometry_mode="conformal"` Wakis instead intersects the STL surfaces with the grid (conformal FIT):

* **PEC walls cut the mesh**: an edge cut by a PEC wall keeps only its length inside the vacuum or dielectric, and a cut face keeps only its area there (a Dey-Mittra-type cut-cell scheme).
* **Dielectric and magnetic interfaces** use effective materials: a mesh edge crossing an interface gets a mix of the two materials, weighted by how the interface cuts it.

This notebook shows the new options on a case with an exact solution, a PEC pillbox cavity with a dielectric liner on its outer wall:

| | |
|---|---|
| `geometry_mode` | `"staircase"` (default) or `"conformal"` |
| `stl_material_types` | `"normal"`, `"pec"` (and `"sibc"`) per STL solid |
| `bg_material_type` | type of the background material |
| `conformal_stabilization` | `"area_floor"` keeps the time step stable with small cut cells |
| `solver.compute_stable_timestep()` | exact stability limit of the time step |
| `solver.plot_mesh_materials()` | the mesh, its materials and the STL geometry in one plane |

In [1]:
import tempfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from scipy.constants import c
from scipy.optimize import brentq
from scipy.special import jn_zeros, jv, yv

from wakis import GridFIT3D, SolverFIT3D, WakeSolver

# Show warnings as 'UserWarning: message', without the source file location.
warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {message}\n"
)

## Geometry and analytic reference

The cavity has radius $R$ and length $L$. A dielectric liner ($\varepsilon_r = 3$) covers the outer wall between $r = b$ and $R$, so the beam on axis only travels through vacuum.

We build it from two cylinders: the full cavity filled with the liner material, and a vacuum cylinder of radius $b$ inside it. Where solids overlap, the solid listed later in `stl_solids` wins. Both cylinders are longer than the simulation domain, so the PEC domain boundaries in $z$ form the flat end plates. The beam enters and leaves through these end plates, which is the usual idealization of infinitely small beam holes.

In [2]:
R, b, L = 0.10, 0.07, 0.10  # cavity radius, liner inner radius, length [m]
eps_liner = 3.0

stl_folder = tempfile.mkdtemp()
stl_solids = {}
for name, radius in (("liner", R), ("vacuum", b)):
    cylinder = pv.Cylinder(
        radius=radius, height=1.2 * L, direction=(0, 0, 1), resolution=200
    )
    stl_solids[name] = f"{stl_folder}/{name}.stl"
    cylinder.triangulate().save(stl_solids[name])

stl_materials = {"liner": [eps_liner, 1.0, 0.0], "vacuum": "vacuum"}

Because the liner is uniform along $z$, the azimuthally symmetric TM modes stay exact Bessel solutions. With $\beta = p\pi/L$, the radial wave numbers are $q_0^2 = k^2 - \beta^2$ in vacuum and $q_1^2 = \varepsilon_r k^2 - \beta^2$ in the liner:

$$E_z = J_0(q_0 r) \quad (r < b), \qquad E_z = J_0(q_1 r)\,Y_0(q_1 R) - Y_0(q_1 r)\,J_0(q_1 R) \quad (b < r < R),$$

which vanishes at the PEC wall. $E_z$ and $H_\varphi \propto (\varepsilon_r / q^2)\, \partial_r E_z$ must be continuous at $r = b$; the roots of this condition are the TM$_{01p}$ frequencies. For TM011, $q_0$ becomes imaginary (the field decays towards the axis); complex Bessel arguments cover that case.

We look at two modes:

* **TM010** has only $E_z$, parallel to both surfaces. It mainly tests the curved PEC wall.
* **TM011** has a radial electric field that crosses the liner surface, so it also tests the effective materials at the dielectric interface.

In [3]:
def f_tm01p(p):
    # Exact TM01p frequency of the lined pillbox.
    beta = p * np.pi / L

    def mismatch(f):
        k = 2 * np.pi * f / c
        q0 = np.sqrt(complex(k**2 - beta**2))
        q1 = np.sqrt(complex(eps_liner * k**2 - beta**2))
        ez_in, hphi_in = jv(0, q0 * b), -jv(1, q0 * b) / q0
        ez_out = jv(0, q1 * b) * yv(0, q1 * R) - yv(0, q1 * b) * jv(0, q1 * R)
        hphi_out = -(eps_liner / q1) * (
            jv(1, q1 * b) * yv(0, q1 * R) - yv(1, q1 * b) * jv(0, q1 * R)
        )
        return (ez_in * hphi_out - hphi_in * ez_out).real

    # Bracket: between the fully filled and the empty cavity
    f_empty = np.hypot(jn_zeros(0, 1)[0] / R, beta) * c / (2 * np.pi)
    f_filled = max(
        f_empty / np.sqrt(eps_liner),
        1.001 * beta * c / (2 * np.pi * np.sqrt(eps_liner)),
    )
    return brentq(mismatch, f_filled, f_empty)


f_analytic = {"TM010": f_tm01p(0), "TM011": f_tm01p(1)}
for mode, f in f_analytic.items():
    print(f"{mode}: {f / 1e6:.2f} MHz")

TM010: 1037.43 MHz
TM011: 1448.24 MHz


## Grid and solver setup

The conformal grid needs to know how each solid takes part in the cut-cell treatment:

* `stl_material_types`: `"normal"` for vacuum, dielectric or magnetic solids (they are kept and get effective materials at their interfaces), `"pec"` for perfect conductors (they cut the mesh). Here both solids are `"normal"`.
* `bg_material_type="pec"`: the background around the solids is a perfect conductor, so the cavity wall at $r = R$ is a PEC wall cutting the mesh. It must match the solver background `bg="pec"`.
* `conformal_stabilization="area_floor"`: explained in the stability section below.

The staircase grid uses the same solids and materials.

In [4]:
def make_solver(
    geometry_mode,
    n_cells=20,
    stabilization="area_floor",
    wake=None,
    solids=stl_solids,
    materials=stl_materials,
):
    # n_cells across the domain (1.1 x the cavity diameter); same cell size in z.
    h = 1.1 * R
    conformal_options = {}
    if geometry_mode == "conformal":
        conformal_options = dict(
            stl_material_types={key: "normal" for key in solids},
            bg_material_type="pec",
            conformal_stabilization=stabilization,
        )
    grid = GridFIT3D(
        -h,
        h,
        -h,
        h,
        -L / 2,
        L / 2,
        n_cells + 1,
        n_cells + 1,
        int(n_cells * L / (2 * h)) + 1,
        stl_solids=solids,
        stl_materials=materials,
        geometry_mode=geometry_mode,
        verbose=0,
        **conformal_options,
    )
    return SolverFIT3D(
        grid,
        wake,
        bg="pec",
        bc_low=["pec"] * 3,
        bc_high=["pec"] * 3,
        use_stl=True,
        dtype=np.float64,
        verbose=0,
    )


solvers = {mode: make_solver(mode) for mode in ("staircase", "conformal")}

Generating grid with 4410 mesh cells...
Initializing Electromagnetic solver...


Generating grid with 4410 mesh cells...


Initializing Electromagnetic solver...


## Inspecting the geometry on the mesh

`solver.plot_mesh_materials()` shows one grid plane with the mesh, the material of every face and the exact STL cross-section (black lines). In FIT, the material of an electric or magnetic DOF belongs to the face its edge pierces, so there are two views:

* `grid_type="primal"`: primal faces, coloured by $\mu_r$. Faces cut by a PEC wall are coloured only in their part inside the cavity (the PEC part is grey), and the mesh edges stop at the wall.
* `grid_type="dual"`: dual faces, coloured by $\varepsilon_r$ (or $\sigma$ with `quantity="sigma"`). Dual faces are never cut; faces whose electric edge lies in PEC are grey.

The plane normal selects the tensor component: in an $x$-normal plane the faces carry $\varepsilon_{r,x}$.

**PEC wall.** The primal $z$-plane: the staircase follows the cell faces, the conformal mesh follows the circle.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(grid_type="primal", normal="z", ax=ax)
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

**Dielectric interface.** The dual $x$-plane next to the axis shows $\varepsilon_{r,x}$. The staircase switches between 1 and 3 from one cell to the next; the conformal mesh gives the edges crossing the liner surface an effective value in between. It depends on where the surface cuts the edge and on its orientation: the field normal to the surface sees the harmonic mean of the two materials (series), the field along it the arithmetic mean (parallel).

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(
        grid_type="dual", normal="x", index=10, quantity="eps", ax=ax
    )
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

Red outlines mark the conformal stabilization (next section). In Jupyter, `interactive=True` adds buttons for the plane normal (and the material) and a slider to move through the planes; it needs an interactive backend such as `%matplotlib widget`:

```python
solvers["conformal"].plot_mesh_materials(grid_type="dual", interactive=True)
```

## Geometry diagnostics

While building a conformal grid, Wakis checks every mesh edge against the STL surfaces and warns about two kinds of edges:

* **Thin feature**: the edge contains material structure thinner than the mesh, e.g. a layer that lies completely between the two edge ends. At most one interface per edge is represented, so the layer is partly or completely lost.
* **Inconsistent**: the STL crossings found on the edge do not match the inside/outside classification of its end points, e.g. at an open (non-watertight) STL surface. These edges keep the staircase material or an ambiguous cut.

The warning only gives the number of edges. `plot_mesh_materials()` shows where they are: orange circles mark thin features, magenta crosses inconsistent edges, and the legend counts them in the plane. We add two flawed geometries to the lined cavity, one for each kind.

**Thin feature: a thin dielectric window.** A dielectric disc ($\varepsilon_r = 3$) across the vacuum region, 0.3 cells thick and placed between two node planes, so no grid node lies inside it.

In [7]:
z_nodes = solvers["conformal"].grid.z
dz = z_nodes[1] - z_nodes[0]
k_window = len(z_nodes) // 2
stl_solids_window = {**stl_solids, "window": f"{stl_folder}/window.stl"}
pv.Cylinder(
    center=(0, 0, z_nodes[k_window] + 0.25 * dz),
    radius=b,
    height=0.3 * dz,
    direction=(0, 0, 1),
    resolution=200,
).triangulate().save(stl_solids_window["window"])

solver_window = make_solver(
    "conformal",
    solids=stl_solids_window,
    materials={**stl_materials, "window": [eps_liner, 1.0, 0.0]},
)

Generating grid with 4410 mesh cells...


129 edges contain material layers thinner than the mesh resolution; at most one interface per edge is represented. Refine the mesh there if these layers matter.


Initializing Electromagnetic solver...


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
solver_window.plot_mesh_materials(grid_type="primal", normal="y", ax=axes[0])
solver_window.plot_mesh_materials(
    grid_type="dual", normal="z", index=k_window + 1, ax=axes[1]
)
fig.tight_layout()

The side view (left, primal $y$-plane) marks the $z$-edges through the window. The dual $z$-plane through the window (right) shows the consequence: the faces pierced by these edges still have $\varepsilon_{r,z} = 1$, so the window is missing from the simulation. Refining the mesh until the window spans at least about one cell fixes this.

**Inconsistent: an STL with a hole.** The inner vacuum cylinder is exported with a small hole in its side wall, as can happen with a damaged CAD export. Wakis warns that the surface is not closed and that edges are inconsistent:

In [9]:
cylinder = pv.Cylinder(
    radius=b, height=1.2 * L, direction=(0, 0, 1), resolution=200
).triangulate()
centers = cylinder.cell_centers().points
angle = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
hole = (
    (np.abs(np.hypot(centers[:, 0], centers[:, 1]) - b) < 1e-3)  # side wall
    & (np.abs(angle - 30) < 12)
    & (np.abs(centers[:, 2]) < 0.03)
)
stl_solids_hole = {**stl_solids, "vacuum": f"{stl_folder}/vacuum_hole.stl"}
cylinder.extract_cells(np.nonzero(~hole)[0]).extract_surface(
    algorithm="dataset_surface"
).save(stl_solids_hole["vacuum"])

solver_hole = make_solver("conformal", solids=stl_solids_hole)

Generating grid with 4410 mesh cells...


115 edges have STL crossings that disagree with the grid point classification and receive no or an ambiguous conformal correction. Check that the STL solids are closed and not tangent to grid points.


Initializing Electromagnetic solver...


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
solver_hole.plot_mesh_materials(grid_type="primal", normal="z", ax=axes[0])
solver_hole.plot_mesh_materials(grid_type="dual", normal="z", ax=axes[1])
fig.tight_layout()

The STL cross-section (black) shows the gap in the vacuum cylinder, and the crosses mark the edges next to it. Without a closed surface, inside and outside are not well defined: in the dual plane (right) some faces of the liner next to the hole get the wrong material. Repair the STL (make it watertight) before simulating.

## Stability of the time step

The explicit leapfrog update is stable only for $\Delta t < 2/\omega_{\max}$, where $\omega_{\max}$ is the highest eigenfrequency of the discrete system. On an uncut grid this is the CFL limit. A face cut by a PEC wall into a tiny piece makes $\omega_{\max}$ very large, and the time step needed for stability would become tiny.

`conformal_stabilization="area_floor"` raises every cut face to at least a fraction (default 0.1) of its full area, and removes edges with less than that fraction left in the cavity. This bounds the time step reduction at the cost of a small, local geometry change. `compute_stable_timestep()` computes the exact stability limit of the assembled solver and compares it with the time step in use:

In [11]:
cases = {
    "staircase": solvers["staircase"],
    "conformal, no stabilization": make_solver("conformal", stabilization=None),
    "conformal, area floor": solvers["conformal"],
}
for name, solver in cases.items():
    result = solver.compute_stable_timestep(verbose=False)
    print(
        f"{name:28s} dt_max / dt_CFL = {result['dt_max'] / result['dt_cfl']:.3f}   "
        f"dt / dt_max = {result['dt_ratio']:.2f}   stable: {result['stable']}"
    )
print(
    f"area floor: {solvers['conformal'].n_area_floored} faces raised, "
    f"{solvers['conformal'].n_edges_eliminated} edges removed"
)

Generating grid with 4410 mesh cells...


Initializing Electromagnetic solver...
staircase                    dt_max / dt_CFL = 1.009   dt / dt_max = 0.50   stable: True
conformal, no stabilization  dt_max / dt_CFL = 0.719   dt / dt_max = 0.70   stable: True
conformal, area floor        dt_max / dt_CFL = 1.011   dt / dt_max = 0.49   stable: True
area floor: 348 faces raised, 120 edges removed


Without stabilization, the cut faces lower the stable time step here to about 70 % of the CFL limit. The solver's default time step (half the CFL limit, `cfln=0.5`) is still below it, but smaller cut faces, for example a surface passing very close to a grid node, lower it further until the default time step diverges. With the area floor, the stable time step stays at the CFL limit of the uncut grid, whatever the geometry.

## Wakefield simulation

We run the same wakefield simulation on both grids: a Gaussian bunch on axis, 30 m of wake. PEC walls make the modes undamped, so the impedance shows a peak at each cavity mode; the side lobes come from cutting the wake after 30 m. We compare the peak positions with the analytic frequencies.

The simulations print progress bars; `%%capture` keeps the notebook short.

In [12]:
%%capture


def run_wake(geometry_mode, n_cells=20):
    folder = tempfile.mkdtemp()
    wake = WakeSolver(
        q=1e-9,
        sigmaz=0.03,
        beta=1.0,
        save=False,
        results_folder=f"{folder}/",
        Ez_file=f"{folder}/Ez.h5",
        verbose=0,
    )
    solver = make_solver(geometry_mode, n_cells, wake=wake)
    solver.wakesolve(wakelength=30.0, plot=False)
    wake.calc_long_Z(samples=20001, fmax=2e9)
    return np.asarray(wake.f), np.asarray(wake.Z)


def peak_error(f, Z, mode):
    # Relative error of the impedance peak next to the analytic mode frequency.
    f_exact = f_analytic[mode]
    window = np.abs(f - f_exact) < 0.15 * f_exact
    return f[window][np.argmax(np.abs(Z[window]))] / f_exact - 1


impedance = {mode: run_wake(mode) for mode in ("staircase", "conformal")}

In [13]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for mode, (f, Z) in impedance.items():
    ax.plot(f / 1e9, np.abs(Z), label=mode)
for name, f_exact in f_analytic.items():
    ax.axvline(f_exact / 1e9, color="k", ls="--", lw=0.8)
    ax.text(f_exact / 1e9, ax.get_ylim()[1] * 0.9, f" {name}", fontsize=9)
ax.set(
    xlabel="f [GHz]",
    ylabel=r"$|Z_\parallel|$ [$\Omega$]",
    yscale="log",
    xlim=(0.8, 1.6),
    ylim=(50, 5e4),
)
ax.legend()

for mode, (f, Z) in impedance.items():
    errors = ", ".join(
        f"{name} {peak_error(f, Z, name) * 100:+.2f} %" for name in f_analytic
    )
    print(f"{mode:10s}: {errors}")

staircase : TM010 -9.15 %, TM011 -6.36 %
conformal : TM010 -0.71 %, TM011 -0.94 %


## Convergence with mesh refinement

The clearest difference shows when refining the mesh. We repeat the simulation on three meshes and plot the frequency errors against the number of cells per cavity radius.

In [14]:
%%capture
n_cells_list = [20, 30, 40]
errors = {
    mode: {name: [] for name in f_analytic} for mode in ("staircase", "conformal")
}
for mode in errors:
    for n_cells in n_cells_list:
        f, Z = impedance[mode] if n_cells == 20 else run_wake(mode, n_cells)
        for name in f_analytic:
            errors[mode][name].append(abs(peak_error(f, Z, name)))

In [15]:
for mode in errors:
    for name in f_analytic:
        print(
            f"{mode:10s} {name}: "
            + ", ".join(f"{100 * e:.2f} %" for e in errors[mode][name])
        )

cells_per_radius = np.array(n_cells_list) / 2.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, name in zip(axes, f_analytic):
    for mode, marker in (("staircase", "s"), ("conformal", "o")):
        ax.loglog(
            cells_per_radius,
            100 * np.array(errors[mode][name]),
            marker=marker,
            label=mode,
        )
    ax.set(title=name, xlabel="cells per radius")
    ax.set_xticks(cells_per_radius, [f"{x:.1f}" for x in cells_per_radius])
    ax.set_xticks([], minor=True)
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel("frequency error [%]")
axes[0].legend()
fig.tight_layout()

staircase  TM010: 9.15 %, 7.86 %, 6.01 %
staircase  TM011: 6.36 %, 4.53 %, 3.60 %
conformal  TM010: 0.71 %, 0.28 %, 0.14 %
conformal  TM011: 0.94 %, 0.45 %, 0.20 %


## Summary

* `geometry_mode="conformal"` represents curved PEC walls and dielectric interfaces inside the mesh cells. On the same mesh the frequency errors are about ten times smaller than with the staircase, and they decrease about with the square of the cell size, while the staircase errors decrease only slowly.
* Each STL solid needs a `stl_material_types` entry: `"normal"` solids get effective materials at their interfaces, `"pec"` solids cut the mesh. `bg_material_type` sets the type of the background and must match the solver background `bg`.
* Keep `conformal_stabilization="area_floor"` for time-domain simulations with PEC cut cells; `compute_stable_timestep()` checks the resulting time step.
* `plot_mesh_materials()` is the quickest way to check how the mesh represents a geometry before running a simulation.
* Check the geometry diagnostics warnings: `plot_mesh_materials()` marks where layers thinner than the mesh are lost and where the STL is not consistent (e.g. not watertight).

Current limitations: each mesh edge is cut at most once (an edge crossing two surfaces is reported in a warning and treated approximately), and the effective materials at dielectric interfaces are diagonal approximations of the full tensor.